# 🌡️ Project 7.1 — Temperature Sensor Priority Queue
**DSA for Mechatronics · Week 07 — Heaps & Priority Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import Counter, defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory floor has many temperature sensors. The control system must always know
which sensors are **hottest** (potential overheating) and which are **coldest** (potential freeze).

A **max-heap** keeps the hottest sensor at the top — O(1) peek, O(log n) insert/extract.
We also implement the heap **from scratch** (array representation) so you can see every sift operation,
and then cross-check with Python's `heapq` module.


## Step 1 — Generate your sensor dataset

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
N_SENSORS    = random.randint(16, 28)
TEMP_MIN     = random.randint(10, 30)
TEMP_MAX     = random.randint(80, 140)
K_HOT        = random.randint(3, 6)     # top-K hottest to report
K_COLD       = random.randint(2, 4)     # bottom-K coldest to report
N_NEW        = random.randint(4, 8)     # new readings arriving after build

SENSOR_ZONES = ["MOTOR_A","MOTOR_B","SPINDLE","COOLANT","BEARING_L",
                "BEARING_R","GEARBOX","COMPRESSOR","PUMP","INVERTER",
                "BRAKE","CLUTCH","FAN","ENCLOSURE","BUS_BAR","CONTACTOR"]

temps   = [random.randint(TEMP_MIN, TEMP_MAX) for _ in range(N_SENSORS)]
zones   = random.choices(SENSOR_ZONES, k=N_SENSORS)
sensors = [(temps[i], f"S{i+1:02d}", zones[i]) for i in range(N_SENSORS)]

print(f"Sensor dataset parameters:")
print(f"  Sensors      : {N_SENSORS}")
print(f"  Temp range   : {TEMP_MIN} – {TEMP_MAX} °C")
print(f"  Top-K hot    : {K_HOT}")
print(f"  Top-K cold   : {K_COLD}")
print(f"  New readings : {N_NEW}")
print()
print(f"  {'ID':<6}  {'Zone':<12}  {'Temp (°C)':>10}")
print(f"  {'─'*6}  {'─'*12}  {'─'*10}")
for t, sid, zone in sensors:
    print(f"  {sid:<6}  {zone:<12}  {t:10d}")


## Step 2 — Build max-heap from scratch (bottom-up heapify, O(n))

In [ ]:
class MaxHeap:
    """
    Max-heap stored as an array.
    Parent of i  → (i-1)//2
    Left child   → 2*i+1
    Right child  → 2*i+2
    Each element is (temp, sensor_id, zone).
    """
    def __init__(self):
        self._data = []

    # ── index helpers ──────────────────────────────────────────────
    def _parent(self, i): return (i - 1) // 2
    def _left(self, i):   return 2 * i + 1
    def _right(self, i):  return 2 * i + 2

    def _swap(self, i, j):
        self._data[i], self._data[j] = self._data[j], self._data[i]

    # ── sift operations ────────────────────────────────────────────
    def _sift_up(self, i):
        """Bubble element at i upward until heap property is restored."""
        while i > 0:
            p = self._parent(i)
            if self._data[i][0] > self._data[p][0]:
                self._swap(i, p)
                i = p
            else:
                break

    def _sift_down(self, i, n=None):
        """Push element at i downward until heap property is restored."""
        if n is None: n = len(self._data)
        while True:
            largest = i
            l, r = self._left(i), self._right(i)
            if l < n and self._data[l][0] > self._data[largest][0]:
                largest = l
            if r < n and self._data[r][0] > self._data[largest][0]:
                largest = r
            if largest == i:
                break
            self._swap(i, largest)
            i = largest

    # ── public API ────────────────────────────────────────────────
    def build(self, items):
        """
        Build heap from list in O(n) — bottom-up heapify.
        Start from last non-leaf (n//2 - 1) and sift down each node.
        """
        self._data = list(items)
        n = len(self._data)
        for i in range(n // 2 - 1, -1, -1):
            self._sift_down(i)

    def push(self, item):
        """Insert new element: append then sift up. O(log n)."""
        self._data.append(item)
        self._sift_up(len(self._data) - 1)

    def pop(self):
        """Extract max: swap root with last, shrink, sift down. O(log n)."""
        if not self._data: return None
        self._swap(0, len(self._data) - 1)
        val = self._data.pop()
        if self._data: self._sift_down(0)
        return val

    def peek(self): return self._data[0] if self._data else None
    def __len__(self):  return len(self._data)

heap = MaxHeap()
heap.build(sensors)

print(f"Max-heap built (bottom-up O(n)):")
print(f"  Heap size      : {len(heap)}")
print(f"  Hottest sensor : {heap.peek()[1]}  {heap.peek()[2]}  {heap.peek()[0]} °C")
print()
print("  Heap array (first 10 elements):")
print(f"  {'Idx':>4}  {'Temp':>6}  {'ID':<6}  Zone")
print(f"  {'─'*4}  {'─'*6}  {'─'*6}  {'─'*12}")
for i, (t, sid, zone) in enumerate(heap._data[:10]):
    parent_str = f"parent={heap._data[heap._parent(i)][0]}°C" if i > 0 else "root"
    print(f"  {i:4d}  {t:6d}  {sid:<6}  {zone:<12}  ({parent_str})")


## Step 3 — Extract top-K hottest and find K coldest

In [ ]:
# Extract top-K hottest (destructive — work on a copy)
import copy
heap_copy = MaxHeap()
heap_copy.build(sensors)

top_k_hot = []
for _ in range(K_HOT):
    top_k_hot.append(heap_copy.pop())

print(f"Top {K_HOT} hottest sensors:")
print(f"  {'Rank':>5}  {'Temp':>6}  {'ID':<6}  Zone")
print(f"  {'─'*5}  {'─'*6}  {'─'*6}  {'─'*12}")
for rank, (t, sid, zone) in enumerate(top_k_hot, 1):
    print(f"  {rank:5d}  {t:6d}  {sid:<6}  {zone}")

# Top-K coldest via heapq (min-heap, no negation needed)
cold_heap = [(t, sid, zone) for t, sid, zone in sensors]
heapq.heapify(cold_heap)

top_k_cold = []
for _ in range(K_COLD):
    top_k_cold.append(heapq.heappop(cold_heap))

print(f"\nTop {K_COLD} coldest sensors:")
print(f"  {'Rank':>5}  {'Temp':>6}  {'ID':<6}  Zone")
print(f"  {'─'*5}  {'─'*6}  {'─'*6}  {'─'*12}")
for rank, (t, sid, zone) in enumerate(top_k_cold, 1):
    print(f"  {rank:5d}  {t:6d}  {sid:<6}  {zone}")


## Step 4 — Insert new readings and re-check top

In [ ]:
# Generate new sensor readings arriving after initial build
new_readings = []
for i in range(N_NEW):
    t    = random.randint(TEMP_MIN, TEMP_MAX + 20)   # some may be hotter than before
    sid  = f"S{N_SENSORS + i + 1:02d}"
    zone = random.choice(SENSOR_ZONES)
    new_readings.append((t, sid, zone))

# Insert into fresh heap
live_heap = MaxHeap()
live_heap.build(sensors)

print(f"Inserting {N_NEW} new sensor readings:")
print(f"  {'ID':<6}  {'Zone':<12}  {'Temp':>6}  {'New top after insert':>22}")
print(f"  {'─'*6}  {'─'*12}  {'─'*6}  {'─'*22}")
for reading in new_readings:
    live_heap.push(reading)
    top = live_heap.peek()
    print(f"  {reading[1]:<6}  {reading[2]:<12}  {reading[0]:6d}  {top[1]} {top[0]}°C")

overall_max = live_heap.peek()
print(f"\n  Final hottest sensor: {overall_max[1]}  {overall_max[2]}  {overall_max[0]} °C")
print(f"  Total sensors tracked: {len(live_heap)}")

# Heap sort verification (in-place, O(n log n))
arr = [t for t, _, _ in sensors]
n   = len(arr)
# build max-heap directly in arr
def sift_down_arr(arr, i, n):
    while True:
        largest = i
        l, r = 2*i+1, 2*i+2
        if l < n and arr[l] > arr[largest]: largest = l
        if r < n and arr[r] > arr[largest]: largest = r
        if largest == i: break
        arr[i], arr[largest] = arr[largest], arr[i]
        i = largest

for i in range(n // 2 - 1, -1, -1):
    sift_down_arr(arr, i, n)
for i in range(n - 1, 0, -1):
    arr[0], arr[i] = arr[i], arr[0]
    sift_down_arr(arr, 0, i)

is_sorted = all(arr[i] <= arr[i+1] for i in range(len(arr)-1))
print(f"\n  Heap-sort verification: {'✅ sorted correctly' if is_sorted else '❌ error'}")
print(f"  Sorted temps: {arr}")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " TEMPERATURE SENSOR PRIORITY QUEUE — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  DATASET" + " "*(W-9) + "║")
print(f"║  {'Sensors':<26}: {N_SENSORS:<26}║")
print(f"║  {'Temp range':<26}: {TEMP_MIN} – {TEMP_MAX} °C{'':<18}║")
print("╠" + "═"*W + "╣")
print("║  HEAP RESULTS" + " "*(W-14) + "║")
t0,s0,z0 = top_k_hot[0]
print(f"║  {'Hottest sensor':<26}: {s0} {z0} {t0}°C{'':<10}║")
tc,sc,zc = top_k_cold[0]
print(f"║  {'Coldest sensor':<26}: {sc} {zc} {tc}°C{'':<10}║")
print(f"║  {'Top-{K_HOT} hot sensors':<26}: {[s for _,s,_ in top_k_hot]}{'':<4}║"[:W+2]+"║")
print(f"║  {'Top-{K_COLD} cold sensors':<26}: {[s for _,s,_ in top_k_cold]}{'':<4}║"[:W+2]+"║")
print("╠" + "═"*W + "╣")
print("║  AFTER NEW READINGS" + " "*(W-20) + "║")
print(f"║  {'New readings inserted':<26}: {N_NEW:<26}║")
om_t,om_s,om_z = overall_max
print(f"║  {'Overall hottest':<26}: {om_s} {om_z} {om_t}°C{'':<10}║")
print(f"║  {'Total sensors tracked':<26}: {len(live_heap):<26}║")
print(f"║  {'Heap-sort correct':<26}: {'YES ✅' if is_sorted else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which heap concept did you find most important, and why?**
Pick one technique (sift-up, sift-down, two-heap median, heapify…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
